# Лабораторная работа 2. Борьба со спамом
## Часть 2. Мешок слов и наивный байесовский классификатор

**Выполнил(а):** Ниязов Никита Алексеевич, Зверев Егор Дмитриевич, группа ИНБб4301

**Что делаем в этой части:**
1. Переводим тексты писем в числовые векторы — представление **«мешок слов»** (`CountVectorizer`).
2. Обучаем **наивный байесовский классификатор** (`MultinomialNB`) из библиотеки scikit-learn.
3. Сравниваем его качество с чёрным списком из части 1 и делаем выводы.

> Перед началом должна быть выполнена часть 1 (`01. Разбор писем и чёрный список.ipynb`): понадобятся тот же датасет и та же функция чтения писем.

## Шаг 1. Подготовка (как в части 1)

Папка `datasets` должна лежать рядом с блокнотом. В Google Colab загрузите датасет так же, как в части 1 — раскомментируйте нужный вариант в ячейке ниже (локально ничего делать не нужно).

In [2]:

 from google.colab import drive
 drive.mount('/content/drive')
 !apt-get install -y p7zip-full > /dev/null
 !7z x -p1 -o/content/ "/content/drive/MyDrive/datasets.zip" -y

# --- Google Colab, вариант Б: загрузка архива в текущую сессию ---
# from google.colab import files
# uploaded = files.upload()          # выберите datasets.zip
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 datasets.zip -y

Mounted at /content/drive

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 39644509 bytes (38 MiB)

Extracting archive: /content/drive/MyDrive/datasets.zip
 40% 4096 Open              --
Path = /content/drive/MyDrive/datasets.zip
Type = zip
Physical Size = 39644509

  0%      9% 894         18% 1799          27% 2676          36% 3456          43% 4246          52% 5094          61% 6041          70% 6996          78% 7872 - datasets/trec07delay/data/inmail.8080                                                 

Шаг 2. Извлечение текста писем и загрузка меток. Написали функцию для извлечения темы и текстового содержимого каждого письма, а также загрузили метки, определяющие, является письмо спамом (spam) или обычным сообщением (ham). Добавили проверку наличия файлов датасета, чтобы убедиться, что данные доступны для обработки.

In [4]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'


def text_email(path):
    """Извлекает тему и текстовое тело письма отдельно."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)

        subject = str(msg.get('Subject') or '')
        parts = []

        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass

        body = ' '.join(parts)
        return subject, body

    except Exception:
        return '', ''


# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))

Меток прочитано: 10000


Шаг 3. Чтение и предварительная обработка писем. Последовательно прочитали все письма из датасета, сохранили темы, тела и соответствующие метки классов в отдельные списки. Исключили письма, содержащие не более двух слов, и проверили корректность загрузки данных на примерах



In [5]:

subjects, bodies, y = [], [], []
empty = 0

filelist = sorted(
    labels,
    key=lambda name: int(name.split('.')[1])
)

for filename in filelist:
    subject, body = text_email(os.path.join(DATA_DIR, filename))

    if len((subject + ' ' + body).split()) <= 2:
        empty += 1
        continue

    subjects.append(subject)
    bodies.append(body)
    y.append(labels[filename])

print(f'Прочитано писем: {len(subjects)}, пропущено пустых: {empty}')
print('Пример темы:', subjects[0][:200])
print('Пример тела:', bodies[0][:200])

assert len(subjects) == len(bodies) == len(y)


Прочитано писем: 9868, пропущено пустых: 132
Пример темы: Generic Cialis, branded quality@ 
Пример тела: 


Шаг 4. Проверка пустых тел писем. Подсчитали количество писем с пустым и непустым телом, чтобы оценить полноту текстовых данных и понять, насколько часто классификатору придётся анализировать письма преимущественно по их теме.

In [6]:

empty_bodies = sum(not body.strip() for body in bodies)

print('Всего писем:', len(bodies))
print('Писем с пустым телом:', empty_bodies)
print('Писем с непустым телом:', len(bodies) - empty_bodies)


Всего писем: 9868
Писем с пустым телом: 4193
Писем с непустым телом: 5675


Шаг 5. Разделение датасета на обучающую и тестовую выборки. Разделили письма на обучающую выборку (70%) и тестовую (30%). Использовали стратификацию, чтобы сохранить примерно одинаковую долю спама в обеих выборках. Обучающая часть нужна для построения модели, а тестовая — для объективной оценки её качества на ранее не использованных данных.

In [7]:

from sklearn.model_selection import train_test_split

# Создаём индексы всех писем
indices = list(range(len(subjects)))

# Разделяем данные на обучение и тестирование
train_idx, test_idx = train_test_split(
    indices,
    train_size=0.7,
    random_state=2,
    stratify=y
)

# Метки классов для обеих выборок
y_train = [y[i] for i in train_idx]
y_test = [y[i] for i in test_idx]

print(f'train: {len(train_idx)} писем, test: {len(test_idx)} писем')

print(
    'Доля спама: train {:.1%}, test {:.1%}'.format(
        sum(v == 'spam' for v in y_train) / len(y_train),
        sum(v == 'spam' for v in y_test) / len(y_test)
    )
)


train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


Шаг 6. Обучение и сравнение моделей с разным весом темы письма. Обучили наивный байесовский классификатор MultinomialNB, используя четыре варианта веса темы: 1, 3, 5 и 10. Для каждого варианта преобразовали текст писем в числовые признаки, обучили модель и оценили её качество с помощью метрик Accuracy, Precision, Recall и F1. Построили матрицы ошибок и сводную таблицу, чтобы определить, насколько увеличение веса темы влияет на обнаружение спама.

In [8]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Сравниваем базовый вес темы, тройной и повышенные веса
weights = [1, 3, 5, 10]

results = []

for weight in weights:
    print('\n' + '=' * 60)
    print(f'ВЕС ТЕМЫ: {weight}')
    print('=' * 60)

    # Формируем текст:
    # тема повторяется weight раз, затем добавляется тело письма
    X = [
        subjects[i] * weight + ' ' + bodies[i]
        for i in range(len(subjects))
    ]

    # Используем одни и те же индексы train и test
    X_train = [X[i] for i in train_idx]
    X_test = [X[i] for i in test_idx]

    # Создаём словарь по обучающей выборке
    vectorizer = CountVectorizer()
    X_train_counts = vectorizer.fit_transform(X_train)
    X_test_counts = vectorizer.transform(X_test)

    # Обучаем классификатор
    model = MultinomialNB()
    model.fit(X_train_counts, y_train)

    # Получаем прогнозы
    y_pred = model.predict(X_test_counts)

    # Вычисляем метрики
    accuracy = accuracy_score(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        labels=['spam', 'ham'],
        output_dict=True,
        zero_division=0
    )

    results.append({
        'Вес темы': weight,
        'Accuracy': accuracy,
        'Precision (spam)': report['spam']['precision'],
        'Recall (spam)': report['spam']['recall'],
        'F1 (spam)': report['spam']['f1-score'],
        'Macro F1': report['macro avg']['f1-score']
    })

    print(f'Размер словаря: {len(vectorizer.vocabulary_)}')

    print('\nОтчёт классификации:')
    print(classification_report(
        y_test,
        y_pred,
        digits=3,
        zero_division=0
    ))

    # Матрица ошибок
    cm = pd.DataFrame(
        confusion_matrix(
            y_test,
            y_pred,
            labels=['spam', 'ham']
        ),
        index=['На самом деле spam', 'На самом деле ham'],
        columns=['Прогноз: spam', 'Прогноз: ham']
    )

    display(cm)

# Сводная таблица
results_df = pd.DataFrame(results)

print('\nСРАВНЕНИЕ ВСЕХ ВАРИАНТОВ')

display(
    results_df.style.format({
        'Accuracy': '{:.3%}',
        'Precision (spam)': '{:.3%}',
        'Recall (spam)': '{:.3%}',
        'F1 (spam)': '{:.3%}',
        'Macro F1': '{:.3%}'
    })
)



ВЕС ТЕМЫ: 1
Размер словаря: 45547

Отчёт классификации:
              precision    recall  f1-score   support

         ham      0.924     0.974     0.948       584
        spam      0.994     0.980     0.987      2377

    accuracy                          0.979      2961
   macro avg      0.959     0.977     0.968      2961
weighted avg      0.980     0.979     0.979      2961



,Прогноз: spam,Прогноз: ham
На самом деле spam,2330,47
На самом деле ham,15,569



ВЕС ТЕМЫ: 3
Размер словаря: 47413

Отчёт классификации:
              precision    recall  f1-score   support

         ham      0.914     0.986     0.949       584
        spam      0.997     0.977     0.987      2377

    accuracy                          0.979      2961
   macro avg      0.955     0.982     0.968      2961
weighted avg      0.980     0.979     0.979      2961



,Прогноз: spam,Прогноз: ham
На самом деле spam,2323,54
На самом деле ham,8,576



ВЕС ТЕМЫ: 5
Размер словаря: 47413

Отчёт классификации:
              precision    recall  f1-score   support

         ham      0.914     0.986     0.949       584
        spam      0.997     0.977     0.987      2377

    accuracy                          0.979      2961
   macro avg      0.955     0.982     0.968      2961
weighted avg      0.980     0.979     0.979      2961



,Прогноз: spam,Прогноз: ham
На самом деле spam,2323,54
На самом деле ham,8,576



ВЕС ТЕМЫ: 10
Размер словаря: 47413

Отчёт классификации:
              precision    recall  f1-score   support

         ham      0.892     0.991     0.939       584
        spam      0.998     0.971     0.984      2377

    accuracy                          0.975      2961
   macro avg      0.945     0.981     0.962      2961
weighted avg      0.977     0.975     0.975      2961



,Прогноз: spam,Прогноз: ham
На самом деле spam,2307,70
На самом деле ham,5,579



СРАВНЕНИЕ ВСЕХ ВАРИАНТОВ


,Вес темы,Accuracy,Precision (spam),Recall (spam),F1 (spam),Macro F1
0,1,97.906%,99.360%,98.023%,98.687%,96.760%
1,3,97.906%,99.657%,97.728%,98.683%,96.788%
2,5,97.906%,99.657%,97.728%,98.683%,96.788%
3,10,97.467%,99.784%,97.055%,98.401%,96.159%


### Вывод по варианту 40 «Тройной вес темы письма»

В ходе работы был проведён эксперимент по оценке влияния веса темы электронного письма на качество классификации спама с использованием алгоритма наивного байесовского классификатора MultinomialNB.

В качестве базового варианта использовалось объединение темы и тела письма по формуле `subject + ' ' + body`. Для выполнения варианта 40 был реализован способ формирования текста `subject * 3 + ' ' + body`, при котором тема письма учитывается три раза. Дополнительно были исследованы веса 5 и 10.

При базовом весе Accuracy составила 97,906%, Precision для класса spam — 99,360%, Recall — 98,023%, а F1-мера — 98,687%. При тройном весе темы Accuracy осталась на уровне 97,906%, Precision увеличилась до 99,657%, Recall снизилась до 97,728%, а F1-мера составила 98,683%. Значение Macro F1 незначительно увеличилось с 96,760% до 96,788%.

При увеличении веса темы до 5 результаты не изменились по сравнению с весом 3. При весе 10 наблюдалось снижение Accuracy до 97,467%, F1 для класса spam до 98,401% и Macro F1 до 96,159%. Количество спам-писем, ошибочно классифицированных как обычные, увеличилось до 70.

Таким образом, тройной вес темы повысил точность положительных предсказаний для класса spam, но не улучшил общую Accuracy. Дальнейшее увеличение веса не привело к улучшению качества классификации и при весе 10 ухудшило ряд основных метрик.